# Replicação do Estudo PyHard para Dataset de Covertype

Este notebook aplica o PyHard no dataset tabular de covertype, seguindo a mesma estrutura de análise do H-CAT.

## Objetivo
Aplicar o PyHard no dataset de covertype e avaliar o desempenho das medidas de Instance Hardness em identificar exemplos perturbados (hard samples).


In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import openml
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

pyhard_path = Path("../pyhard").resolve()
if str(pyhard_path) not in sys.path:
    sys.path.insert(0, str(pyhard_path))

from pyhard.measures import ClassificationMeasures


In [ ]:
TOTAL_RUNS = 3
SEED = 0

# PyHard usa matriz N×N (Gower); subamostra para caber em memória.
HCAT_POOL_SIZE = 50000
PYHARD_SAMPLE_SIZE = 20000

HARDNESS_TYPES = ["uniform", "asymmetric", "adjacent", "instance", "atypical"]
PROPORTIONS = [0.1, 0.3, 0.5]

# Covertype tem classes 1-7; após LabelEncoder, a rule matrix usa índices 0-6.
RULE_MATRIX_COVERTYPE = {4: [5], 1: [0], 0: [6], 6: [0], 2: [3], 5: [2], 3: [2]}
ATYPICAL_FEATURE = "Elevation"


In [ ]:
dataset = openml.datasets.get_dataset(1596)
X, y, _, attribute_names = dataset.get_data(
    dataset_format="dataframe",
    target=dataset.default_target_attribute,
)
df = pd.DataFrame(X, columns=attribute_names).dropna()
df["y"] = LabelEncoder().fit_transform(y.loc[df.index])
df = df.select_dtypes(include=[np.number]).dropna()

if len(df) > HCAT_POOL_SIZE:
    df = df.sample(HCAT_POOL_SIZE, random_state=SEED).reset_index(drop=True)
if len(df) > PYHARD_SAMPLE_SIZE:
    df = df.sample(PYHARD_SAMPLE_SIZE, random_state=SEED).reset_index(drop=True)


In [ ]:
def uniform_mislabeling(labels, p, seed=None):
    rng = np.random.default_rng(seed)
    labels = np.asarray(labels, dtype=int)
    n_flips = int(round(p * len(labels)))
    flip_idx = rng.choice(len(labels), size=n_flips, replace=False)
    classes = np.unique(labels)
    flipped = labels.copy()
    for i in flip_idx:
        flipped[i] = rng.choice(classes[classes != labels[i]])
    return flip_idx, flipped


def asymmetric_mislabeling(labels, p, n_classes, seed=None):
    rng = np.random.default_rng(seed)
    labels = np.asarray(labels, dtype=int)
    n_flips = int(round(p * len(labels)))

    noise = np.zeros((n_classes, n_classes))
    for i in range(n_classes):
        off = rng.dirichlet(np.ones(n_classes - 1))
        noise[i] = np.insert(off, i, 0)
    noise /= noise.sum(axis=1, keepdims=True)

    flip_idx = rng.choice(len(labels), size=n_flips, replace=False)
    flipped = labels.copy()
    for i in flip_idx:
        flipped[i] = rng.choice(n_classes, p=noise[labels[i]])
    return flip_idx, flipped


def adjacent_mislabeling(labels, p, n_classes, seed=None):
    rng = np.random.default_rng(seed)
    labels = np.asarray(labels, dtype=int)
    n_flips = int(round(p * len(labels)))
    flip_idx = rng.choice(len(labels), size=n_flips, replace=False)
    flipped = labels.copy()
    for i in flip_idx:
        direction = rng.choice([-1, 1])
        flipped[i] = (labels[i] + direction) % n_classes
    return flip_idx, flipped


def instance_mislabeling(labels, flip_idx, rule_matrix, seed=None):
    rng = np.random.default_rng(seed)
    labels = np.asarray(labels, dtype=int)
    flipped = labels.copy()
    for i in flip_idx:
        candidates = rule_matrix.get(int(labels[i]), [])
        if candidates:
            flipped[i] = rng.choice(candidates)
    return flipped


def apply_atypical_perturbation(df, p, feature, seed=None):
    rng = np.random.default_rng(seed)
    n = len(df)
    flip_idx = rng.choice(n, size=int(round(p * n)), replace=False)
    df = df.copy()
    col = df[feature].values.astype(float)
    scale = col.std() * 3 if col.std() > 0 else 1.0
    df.loc[df.index[flip_idx], feature] = col[flip_idx] + rng.normal(scale, scale / 2, size=len(flip_idx))
    return flip_idx, df


In [ ]:
def run_experiment(hardness_type, proportion, run_number, seed, df_original):
    np.random.seed(seed)
    df = df_original.copy()
    labels = np.asarray(df["y"].values, dtype=int)
    n_classes = len(np.unique(labels))
    flag_ids = None

    if hardness_type == "uniform":
        flag_ids, flipped = uniform_mislabeling(labels, proportion, seed=seed)
        df["y"] = flipped
    elif hardness_type == "asymmetric":
        flag_ids, flipped = asymmetric_mislabeling(labels, proportion, n_classes, seed=seed)
        df["y"] = flipped
    elif hardness_type == "adjacent":
        flag_ids, flipped = adjacent_mislabeling(labels, proportion, n_classes, seed=seed)
        df["y"] = flipped
    elif hardness_type == "instance":
        rule_matrix = RULE_MATRIX_COVERTYPE if RULE_MATRIX_COVERTYPE is not None else {
            int(c): [int(k) for k in np.unique(labels) if k != c] for c in np.unique(labels)
        }
        flag_ids = np.random.choice(len(df), int(len(df) * proportion), replace=False)
        df["y"] = instance_mislabeling(labels, flag_ids, rule_matrix, seed=seed)
    elif hardness_type == "atypical":
        feature = ATYPICAL_FEATURE if ATYPICAL_FEATURE in df.columns else (
            [c for c in df.columns if c != "y" and df[c].dtype.kind in "fi"][0]
        )
        flag_ids, df = apply_atypical_perturbation(df, proportion, feature, seed=seed)

    df["y"] = df["y"].astype(int)
    is_perturbed = np.zeros(len(df), dtype=int)
    is_perturbed[flag_ids] = 1

    measures_list = [
        "kDN", "DCP", "TD_P", "TD_U", "CL", "CLD", "MV", "CB", "N1",
        "N2", "LSC", "LSR", "Harmfulness", "Usefulness", "F1",
    ]
    cm = ClassificationMeasures(df, target_col="y")
    scores = {}
    for measure in measures_list:
        try:
            fn = getattr(cm, measure)
            scores[measure] = fn()
        except Exception:
            scores[measure] = np.full(len(df), np.nan)

    from sklearn.metrics import average_precision_score, roc_auc_score
    metrics = {}
    for measure, values in scores.items():
        if np.all(np.isnan(values)) or np.nanstd(values) == 0:
            metrics[measure] = {"auc_roc": np.nan, "auc_prc": np.nan}
            continue
        vals = np.where(np.isnan(values), np.nanmedian(values), values)
        roc_dir = roc_auc_score(is_perturbed, vals)
        roc_inv = roc_auc_score(is_perturbed, -vals)
        if roc_dir >= roc_inv:
            metrics[measure] = {
                "auc_roc": float(roc_dir),
                "auc_prc": float(average_precision_score(is_perturbed, vals)),
            }
        else:
            metrics[measure] = {
                "auc_roc": float(roc_inv),
                "auc_prc": float(average_precision_score(is_perturbed, -vals)),
            }

    return {
        "hardness_type": hardness_type,
        "proportion": proportion,
        "run": run_number,
        "seed": seed,
        "metrics": metrics,
        "raw_scores": scores,
        "is_perturbed": is_perturbed,
        "flag_ids": flag_ids,
    }


In [ ]:
all_results = []
for hardness_type in HARDNESS_TYPES:
    for prop in PROPORTIONS:
        for run in range(1, TOTAL_RUNS + 1):
            seed = SEED + (run - 1)
            try:
                result = run_experiment(hardness_type, prop, run, seed, df)
            except Exception:
                import traceback
                traceback.print_exc()
                continue
            if result is not None:
                all_results.append(result)


## 8. Análise e Visualização dos Resultados



In [ ]:
results_list = []
for result in all_results:
    for method, metrics in result["metrics"].items():
        results_list.append({
            "hardness_type": result["hardness_type"],
            "proportion": result["proportion"],
            "run": result["run"],
            "method": method,
            "auc_roc": metrics.get("auc_roc", np.nan),
            "auc_prc": metrics.get("auc_prc", np.nan),
        })
results_df = pd.DataFrame(results_list)


In [ ]:
OUTPUT_DIR = Path("..") / "data" / "comparison_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV = OUTPUT_DIR / "nb03_pyhard_covertype_results.csv"

results_to_save = results_df.rename(columns={"method": "measure"}).copy()
results_to_save.insert(0, "dataset", "covertype")
results_to_save.insert(1, "method_family", "pyhard")
results_to_save = results_to_save[[
    "dataset", "method_family", "hardness_type", "proportion", "run", "measure",
    "auc_roc", "auc_prc",
]]
results_to_save.to_csv(OUTPUT_CSV, index=False)


In [ ]:
summary = (
    results_df
    .groupby(["hardness_type", "proportion", "method"])
    .agg({"auc_roc": ["mean", "std"], "auc_prc": ["mean", "std"]})
    .round(4)
)
summary


In [ ]:
for hardness_type in HARDNESS_TYPES:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    subset = results_df[results_df["hardness_type"] == hardness_type]
    for i, metric in enumerate(["auc_roc", "auc_prc"]):
        ax = axes[i]
        for method in subset["method"].unique():
            md = subset[subset["method"] == method]
            mean = md.groupby("proportion")[metric].mean()
            std = md.groupby("proportion")[metric].std()
            ax.errorbar(mean.index, mean.values, yerr=std.values, label=method, marker="o", capsize=3, alpha=0.7)
        ax.set_xlabel("Proporção de perturbação")
        ax.set_ylabel(metric.upper().replace("_", "-"))
        ax.set_title(f"{metric.upper().replace('_', '-')} — {hardness_type}")
        ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)
        ax.set_ylim([0.4, 1.0])
    plt.tight_layout()
    plt.show()


In [ ]:
comparison_data = []
for hardness_type in HARDNESS_TYPES:
    for prop in PROPORTIONS:
        subset = results_df[
            (results_df["hardness_type"] == hardness_type)
            & (results_df["proportion"] == prop)
        ]
        for method, auc_prc in subset.groupby("method")["auc_prc"].mean().items():
            comparison_data.append({
                "hardness_type": hardness_type,
                "proportion": prop,
                "method": method,
                "mean_auc_prc": auc_prc,
            })

comparison_df = pd.DataFrame(comparison_data)
pivot_table = comparison_df.pivot_table(
    values="mean_auc_prc",
    index=["hardness_type", "method"],
    columns="proportion",
    aggfunc="mean",
)
pivot_table.round(4)


In [ ]:
for hardness_type in HARDNESS_TYPES:
    subset = results_df[results_df["hardness_type"] == hardness_type]
    ranking = subset.groupby("method")["auc_prc"].agg(["mean", "std"]).sort_values("mean", ascending=False)
    print(f"\n{hardness_type} (AUC-PRC médio ± std):")
    print(ranking.round(4))


## 12. Comparação com H-CAT

Se os resultados do H-CAT estiverem disponíveis, podemos comparar:


In [ ]:
try:
    hcat_df = pd.read_csv(
        Path("..") / "data" / "comparison_results" / "hcat_diabetes_results.csv"
    )
    print("H-CAT AUC-PRC médio:", round(hcat_df["auc_prc"].mean(), 4))
    print("PyHard AUC-PRC médio:", round(results_df["auc_prc"].mean(), 4))
except FileNotFoundError:
    print("Resultados H-CAT não encontrados (rode antes o notebook 01 ou 05).")
